In [1]:
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
import numpy as np

# =========================
# PARAMETER
# =========================
n = 3                      # jumlah qubit (ganti ke 22 di HPC)
target = "101"              # state target
iterations = 1              # jumlah iterasi Grover

# =========================
# ORACLE
# =========================
def oracle_circuit(n, target):
    qc = QuantumCircuit(n)
    
    # X untuk bit 0
    for i, bit in enumerate(target):
        if bit == "0":
            qc.x(i)
    
    # Multi-controlled Z (via H-MCX-H)
    qc.h(n-1)
    qc.mcx(list(range(n-1)), n-1)
    qc.h(n-1)
    
    # Kembalikan X
    for i, bit in enumerate(target):
        if bit == "0":
            qc.x(i)
            
    return qc


# =========================
# DIFFUSION OPERATOR
# =========================
def diffusion_circuit(n):
    qc = QuantumCircuit(n)
    
    qc.h(range(n))
    qc.x(range(n))
    
    qc.h(n-1)
    qc.mcx(list(range(n-1)), n-1)
    qc.h(n-1)
    
    qc.x(range(n))
    qc.h(range(n))
    
    return qc


# =========================
# GROVER CIRCUIT
# =========================
qc = QuantumCircuit(n)

# Superposition awal
qc.h(range(n))

# Iterasi Grover
for _ in range(iterations):
    qc.compose(oracle_circuit(n, target), inplace=True)
    qc.compose(diffusion_circuit(n), inplace=True)

# Simpan statevector
qc.save_statevector()

# =========================
# SIMULASI
# =========================
sim = AerSimulator(method="statevector")
compiled = transpile(qc, sim)
result = sim.run(compiled).result()

statevector = result.get_statevector()

# =========================
# OUTPUT AMPLITUDO KOMPLEKS
# =========================
print("=== Statevector Amplitudes ===\n")

for i, amp in enumerate(statevector):
    basis_state = format(i, f"0{n}b")
    print(f"|{basis_state}> : "
          f"Real={amp.real:.6f}, "
          f"Imag={amp.imag:.6f}, "
          f"|Amp|={abs(amp):.6f}, "
          f"Prob={abs(amp)**2:.6f}")

=== Statevector Amplitudes ===

|000> : Real=-0.176777, Imag=-0.000000, |Amp|=0.176777, Prob=0.031250
|001> : Real=-0.176777, Imag=-0.000000, |Amp|=0.176777, Prob=0.031250
|010> : Real=-0.176777, Imag=-0.000000, |Amp|=0.176777, Prob=0.031250
|011> : Real=-0.176777, Imag=-0.000000, |Amp|=0.176777, Prob=0.031250
|100> : Real=-0.176777, Imag=-0.000000, |Amp|=0.176777, Prob=0.031250
|101> : Real=-0.883883, Imag=-0.000000, |Amp|=0.883883, Prob=0.781250
|110> : Real=-0.176777, Imag=-0.000000, |Amp|=0.176777, Prob=0.031250
|111> : Real=-0.176777, Imag=-0.000000, |Amp|=0.176777, Prob=0.031250


/tmp/ipykernel_3304845/2537256198.py:85: DeprecationWarning: The return type of saved statevectors has been changed from a `numpy.ndarray` to a `qiskit.quantum_info.Statevector` as of qiskit-aer 0.10. Accessing numpy array attributes is deprecated and will result in an error in a future release. To continue using saved result objects as arrays you can explicitly cast them using  `np.asarray(object)`.
  for i, amp in enumerate(statevector):


In [3]:
statevector[1]

np.complex128(-0.17677669529663706-4.32978028117747e-17j)